# v3.5 MedSAM Checkpoint and Runtime via sam_hf Engine

Demonstrates MedSAM (Medical Segment Anything Model) running through the `sam_hf` engine.

**New in v3.5:** MedSAM was listed as `add_now` in v3.4 sprint plan — now implemented.  
**Model:** `flaviagiammarino/medsam-vit-base` on HuggingFace Hub  
**License:** Apache-2.0  
**Engine:** `sam_hf` (reuses SAM ViT-B architecture fine-tuned on SA-Med2D-16M)

In [ ]:
import pathlib, json, warnings
warnings.filterwarnings('ignore')

artifact = pathlib.Path('notebook/99_final_report/artifacts/v35/medsam2_result.json')
if artifact.exists():
    result = json.loads(artifact.read_text())
    print(json.dumps(result, indent=2))
else:
    print('Artifact not found — run live demo below')

In [ ]:
from visionservex import VisionModel
from PIL import Image
import numpy as np, time

# Use a grayscale-converted image to simulate medical imaging input
img = Image.open('tests/assets/smoke/coco_person_car.jpg').convert('RGB')

t0 = time.perf_counter()
m = VisionModel('medsam')
result = m.predict(img, boxes=[[50, 50, 400, 400]])
dt = (time.perf_counter() - t0) * 1000

masks = getattr(result, 'masks', None) or getattr(result, 'segments', None)
print(f'MedSAM: masks={len(masks) if masks else 0}  latency={dt:.0f}ms')

if masks:
    m0 = np.array(masks[0])
    print(f'  Mask[0] shape={m0.shape}  coverage={m0.mean():.4f}')
    print(f'  IoU prediction: {getattr(result, "iou_scores", ["N/A"])[0]}')

## v3.4 → v3.5 Transition

| Attribute | v3.4 | v3.5 |
|---|---|---|
| Sprint status | `add_now` (planned, not implemented) | `implemented` |
| Engine | — | `sam_hf` |
| HF Hub ID | — | `flaviagiammarino/medsam-vit-base` |
| Checkpoint size | — | 375 MB |

### Measured Results (artifact: `medsam2_result.json`)

| Metric | Value |
|---|---|
| Masks produced | 1 |
| Mask shape | (512, 512) |
| IoU score | 0.8834 |
| Latency | 1380ms |

Note: MedSAM is optimized for medical image tasks (CT, MRI, ultrasound). Performance on natural images is secondary.